# Bronze - YouTube Trending

## Objetivo

Realizar a ingestão e validação dos arquivos públicos de tendências
do YouTube, com foco na identificação de dados do Brasil no ano de 2025.

## Arquivos de origem

- youtube_trending_videos_global.parquet
- youtube_trending_videos_global_daily.parquet

## Estratégia

Os arquivos originais serão mantidos no Volume do Unity Catalog,
sem alteração, caracterizando a camada Bronze física.

Nesta etapa serão avaliados:

- estrutura dos arquivos;
- cobertura temporal;
- presença de registros brasileiros;
- existência de categorias de conteúdo;
- métricas de visualização e interação;
- adequação dos dados ao período de 2025.

A transformação, padronização e recorte do período ocorrerão
posteriormente na camada Silver.

In [0]:
from pyspark.sql import functions as F

In [0]:
caminho_base = "/Volumes/workspace/mvp_bronze/raw_files/youtube_2025"

display(
    dbutils.fs.ls(caminho_base)
)

In [0]:
caminho_global = (
    "/Volumes/workspace/mvp_bronze/raw_files/youtube_2025/"
    "youtube_trending_videos_global.parquet"
)

caminho_daily = (
    "/Volumes/workspace/mvp_bronze/raw_files/youtube_2025/"
    "youtube_trending_videos_global_daily.parquet"
)

print("GLOBAL:")
print(caminho_global)

print("\nDAILY:")
print(caminho_daily)

In [0]:
df_global = spark.read.parquet(caminho_global)

df_daily = spark.read.parquet(caminho_daily)

In [0]:
print("=== ARQUIVO GLOBAL ===")
print("Quantidade de colunas:", len(df_global.columns))

for coluna in df_global.columns:
    print(coluna)

In [0]:
df_global.printSchema()

In [0]:
display(
    df_global.limit(10)
)

In [0]:
print("=== ARQUIVO DAILY ===")
print("Quantidade de colunas:", len(df_daily.columns))

for coluna in df_daily.columns:
    print(coluna)

In [0]:
df_daily.printSchema()

In [0]:
display(
    df_daily.limit(10)
)

In [0]:
def localizar_colunas(df):
    colunas = df.columns

    pais = [
        c for c in colunas
        if "country" in c.lower()
    ]

    data = [
        c for c in colunas
        if "date" in c.lower()
    ]

    categoria = [
        c for c in colunas
        if "category" in c.lower()
    ]

    video = [
        c for c in colunas
        if "video_id" in c.lower()
    ]

    views = [
        c for c in colunas
        if "view" in c.lower()
    ]

    likes = [
        c for c in colunas
        if "like" in c.lower()
    ]

    comentarios = [
        c for c in colunas
        if "comment" in c.lower()
    ]

    print("País:", pais)
    print("Datas:", data)
    print("Categorias:", categoria)
    print("Video ID:", video)
    print("Views:", views)
    print("Likes:", likes)
    print("Comentários:", comentarios)

In [0]:
print("=== GLOBAL ===")
localizar_colunas(df_global)

In [0]:
print("=== DAILY ===")
localizar_colunas(df_daily)

In [0]:
display(
    df_global
    .select("video_trending_country")
    .distinct()
    .orderBy("video_trending_country")
)

In [0]:
df_global_br = (
    df_global
    .filter(
        F.upper(
            F.trim(F.col("video_trending_country"))
        ).isin("BR", "BRA", "BRAZIL")
    )
)

In [0]:
display(
    df_global_br
    .select(
        "video_id",
        "video_trending_country",
        "video_trending__date"
    )
    .limit(20)
)

In [0]:
df_global_br_datas = (
    df_global_br
    .withColumn(
        "_data_texto_normalizada",
        F.regexp_replace(
            F.col("video_trending__date").cast("string"),
            r"\.",
            "-"
        )
    )
    .withColumn(
        "_data_validacao",
        F.expr(
            "try_cast(_data_texto_normalizada as date)"
        )
    )
)

In [0]:
display(
    df_global_br_datas
    .agg(
        F.min("_data_validacao").alias("data_minima"),
        F.max("_data_validacao").alias("data_maxima")
    )
)

In [0]:
display(
    df_global_br_datas
    .filter(
        F.col("_data_validacao").isNotNull()
    )
    .groupBy(
        F.year("_data_validacao").alias("ano")
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy("ano")
)

In [0]:
df_global_br_2025 = (
    df_global_br_datas
    .filter(
        F.year("_data_validacao") == 2025
    )
)

In [0]:
display(
    df_global_br_2025
    .agg(
        F.count("*").alias("registros_2025"),
        F.countDistinct("video_id").alias("videos_unicos_2025")
    )
)

In [0]:
display(
    df_global_br_2025
    .groupBy(
        F.month("_data_validacao").alias("mes")
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy("mes")
)

In [0]:
display(
    df_global_br_datas
    .select(
        "video_trending__date",
        "_data_texto_normalizada",
        "_data_validacao"
    )
    .limit(20)
)

In [0]:
display(
    df_global_br_2025
    .groupBy(
        F.month("_data_validacao").alias("mes")
    )
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy("mes")
)

In [0]:
campos_desejados = [
    "video_id",
    "video_trending_country",
    "video_trending__date",
    "video_category_id",
    "video_view_count",
    "video_like_count",
    "video_comment_count"
]

for campo in campos_desejados:
    print(
        campo,
        "->",
        "OK" if campo in df_global.columns
        else "NÃO ENCONTRADO"
    )

In [0]:
display(
    df_global_br_2025
    .groupBy("video_category_id")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("video_id").alias("videos_unicos")
    )
    .orderBy(F.desc("registros"))
)

In [0]:
display(
    df_global_br_2025
    .agg(
        F.count("*").alias("total_registros"),

        F.sum(
            F.when(F.col("video_id").isNull(), 1).otherwise(0)
        ).alias("video_id_nulo"),

        F.sum(
            F.when(F.col("video_category_id").isNull(), 1).otherwise(0)
        ).alias("categoria_nula"),

        F.sum(
            F.when(F.col("_data_validacao").isNull(), 1).otherwise(0)
        ).alias("data_invalida")
    )
)

## Decisão da fonte

Após a validação dos arquivos disponíveis, o arquivo
`youtube_trending_videos_global.parquet` foi selecionado como fonte
principal do YouTube para o MVP.

A fonte apresentou registros brasileiros nos anos de 2024, 2025 e 2026.

Para o ano de referência de 2025 foram identificados:

- 72.592 registros de tendência;
- 32.370 vídeos únicos.

O arquivo `youtube_trending_videos_global_daily.parquet` foi preservado
no Volume como fonte complementar, mas não será utilizado no pipeline
analítico principal para evitar redundância de processamento.

In [0]:
inventario = [
    (
        "youtube_trending_videos_global.parquet",
        caminho_global,
        "PRINCIPAL",
        "Arquivo histórico utilizado no pipeline do MVP"
    ),
    (
        "youtube_trending_videos_global_daily.parquet",
        caminho_daily,
        "COMPLEMENTAR",
        "Arquivo preservado, mas não utilizado no pipeline principal"
    )
]

df_inventario_youtube = (
    spark.createDataFrame(
        inventario,
        [
            "arquivo",
            "caminho_volume",
            "status_uso",
            "descricao"
        ]
    )
    .withColumn(
        "data_ingestao",
        F.current_timestamp()
    )
)

In [0]:
(
    df_inventario_youtube.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.mvp_bronze.youtube_inventario_fontes"
    )
)

In [0]:
%sql

SHOW TABLES IN workspace.mvp_bronze;